## 07. VulGNN pipeline

This notebook walks into the VulGNN pipeline. VulGNN is a Graph Neural Network trained on the output of Joern, after being processed and normalized. The bugfinder has all the necessary steps from extracting the dataset, generate the Code Property Grapy, normalize it and transform it in an input graph for the Pytorch model implementation. 

In [1]:
# Specific instruction to run the notebooks from a sub-folder.
import sys
sys.path.append("..")

Setting imports and logging level

In [2]:
import logging
from bugfinder.settings import LOGGER

from bugfinder.base.dataset import CodeWeaknessClassificationDataset as Dataset
from bugfinder.processing.cleaning.replace_litterals import ReplaceLitterals
from bugfinder.processing.cleaning.remove_comments import RemoveComments

from bugfinder.processing.dataset.extract import ExtractSampleDataset

from bugfinder.processing.joern.v2010 import JoernProcessing as Joern2010Processing
from bugfinder.processing.tokenizers.normalize_cpgs import NormalizeCodePropertyGraph
from bugfinder.processing.tokenizers.combine_cpgs import CombineDotFiles
from bugfinder.features.extraction.transformers.embeddings import TransformerEmbeddings

In [3]:
# Setup logging to only output INFO level messages
LOGGER.setLevel(logging.INFO)

In [4]:
# Dataset directories (DO NOT EDIT)
orig_dataset_path = "../data/ai-dataset_orig/"
sample_dataset_path = "../data/ai-dataset_sample/"
cpg_dataset_path = "../data/ai-dataset_sample/cpgs/"

In [5]:
import subprocess
from os import listdir
from os.path import isdir, join

force_download = False  # Change to True if the dataset has been tampered with
download_dir = join(orig_dataset_path, "bad")
need_download = (not isdir(download_dir) or len(listdir(download_dir)) != 6507)

if need_download or force_download:
    LOGGER.info("Downloading dataset...")
    subprocess.run("../scripts/setup_ai_dataset.sh")

For testing purposes, only a smalll subset of the dataset will be used to demonstrate how the pipeline works. You can choose the size of it by changing the `sample_nb` variable.

In [6]:
sample_nb = 40

orig_dataset = Dataset(orig_dataset_path)

orig_dataset.queue_operation(
    ExtractSampleDataset,
    {"to_path": sample_dataset_path, "sample_nb": sample_nb, "force": True},
)

orig_dataset.process()

[2024-07-23 00:12:47][INFO] Dataset initialized in 114ms.
[2024-07-23 00:12:47][INFO] Operation queue validated in 0ms.
[2024-07-23 00:12:47][INFO] Running operation 1/1 (bugfinder.processing.dataset.extract.ExtractSampleDataset)...
[2024-07-23 00:12:47][INFO] Dataset extraction succeeded.
[2024-07-23 00:12:47][INFO] 1 operations run in 15ms.


<DatasetQueueRetCode.OK: 0>

### Pre-processing

In this pipeline, it's not necessary to do most of the cleaning steps used in others. Since Joern v2.0 supports C++ files it's not necessary to remove them from the dataset, and since the normalization will be done in the Code Property Graph itself, it's best to avoid this step right here.

Even the `ReplaceLitterals` step it's not really necessary here, although recommended.

In [7]:
sample_dataset = Dataset(sample_dataset_path)

sample_dataset.queue_operation(ReplaceLitterals)
sample_dataset.queue_operation(RemoveComments)

sample_dataset.process()

[2024-07-23 00:13:01][INFO] Dataset initialized in 2ms.
[2024-07-23 00:13:01][INFO] Operation queue validated in 0ms.
[2024-07-23 00:13:01][INFO] Running operation 1/1 (bugfinder.processing.cleaning.replace_litterals.ReplaceLitterals)...
[2024-07-23 00:13:01][INFO] Litterals successfully replaced.
[2024-07-23 00:13:01][INFO] 1 operations run in 46ms.


<DatasetQueueRetCode.OK: 0>

### Executing Joern

This step executes the most recent version of Joern to extract the Code Property Graph. In this step, the CPG is generated and saved in a binary file, and then exported to the desired output.

The possible inputs/outputs for this step are:
- language: defaults to C, Python/Java and other languages are supported
- repr_type: all|ast|cdg|cfg|cpg14|ddg|pdg, defaults to CPG. When the CPG is generated, several overlays are created and you can extract only certain graphs if you need.
- output_format: dot|graphml|graphson|neo4jcsv|json, defaults to dot. It's recommended to export the CPG in DOT format: even though it's possible to use the listed outputs, any export except dot only works with `repr_type` setted to all. For example, if you need only the Data Dependency Graph, you will have to export it as a DOT file. JSON is supported by an external scala script.

On the VulGNN pipeline, the `repr_type` used is the CPG and the `output_format` is the DOT file. After finishing the execution, each processed sample will be exported into a `cpgs` folder inside the path used as an input for the bugfinder.

In [ ]:
sample_dataset.queue_operation(Joern2010Processing, 
                                {"language": "c", 
                                "repr_type": "cpg", 
                                "output_format": "dot"})

sample_dataset.process()


### Normalizing CPGs

After creating and extracting the CPGs into several DOT files, it's time to normalize them. The normalization process does the following:
- Run a sed command to replace <> by "" on the nodes representation. This is done because the normalization process involves transforming the DOT files' content into a networkx graph, and by keeping <> around code it bugs out how networkx processes the graph.
- Removes bugged HTML tags such as `&lt;` and `&gt;`
- Replaces variables and functions by a generic token (VAR/FUN)

After the normalization is done, it's time to combine the different DOT files in a single one. That removes a lot of redundant information related to repeated edges/nodes.

In [ ]:
cpg_dataset = Dataset(cpg_dataset_path)

cpg_dataset.queue_operation(NormalizeCodePropertyGraph)
cpg_dataset.queue_operation(CombineDotFiles)

cpg_dataset.process()

### Generating embeddings

After the normalization step, the result is a single DOT file containing the normalized representation of the CPG. This step is responsible to process this CPG using a networkx graph intermediate representation, and generate embeddings using CodeBERT. The result is a `torch` Data object which is serialized in a `.pkl` file and saved in the same folder as the dot file.

In [ ]:
cpg_dataset.queue_operation(TransformerEmbeddings)

cpg_dataset.process()

### Optional: Process DiverseVul

The notebook uses the standard `ai-dataset`, If you want to run the pipeline on the bigger diversevul dataset, execute the function below to process the JSON file containing the test cases, then point the new path before starting executing the rest

In [ ]:
data_file_path = '../data/diversevul'
json_file = 'diversevul_20230702.json'
print(os.path.abspath(os.path.join(data_file_path, json_file)))

In [ ]:
import json

def load_json_file(json_file_path):
    try:
        with open(json_file_path, "r") as json_file:
            data = []
            lines = json_file.read().splitlines()
                
            for line in lines:
                data.append(json.loads(line))
    
            return data
    except FileNotFoundError:
        print("File not found.")
        return None
    except json.JSONDecodeError as e:
        print(e)
        return None

In [ ]:
loaded_data = load_json_file(os.path.abspath(json_file_path))

In [ ]:
test = os.path.abspath(data_file_path)

counter = 0
classes = {0: "good", 1: "bad"}

for data in loaded_data:
    folder = os.path.join(os.path.abspath(test), classes[data["target"]])
    tail = str(data["project"]) + '-' + str(data["hash"])

    dir_path = (os.path.join(folder, tail))
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

    filepath = os.path.join(dir_path, (tail + '.cpp'))
    with open(filepath, "w") as f:
        f.write(data['func'])

    counter += 1

    if counter >= 20:
        break